# PyTorch NN Basics - Build and Train a Model

<!-- Provenance: FROM-OLD Frameworks/PyTorch_Exercises.ipynb cell ba37ca17. Change: retheme intro from CV-startup/MNIST to the support-team chatbot arc; drop Conv2d bullet; point forward to B7 and C9. -->

*ML & NLP course - Data Trainers LLC - Axel Sirota*

## From tensors to a trained model

In B4 you turned text into tensors and met autograd: `.backward()` and `.grad`. In B5 you turned
text into geometry: every support ticket is now a fixed-size float vector, and you measured a
LogisticRegression baseline on those vectors. This notebook is the assembly manual. You will snap
the tensor bricks into a small neural network and train it end to end.

You will not need a GPU or a big dataset. We train on a small toy feature matrix that stands in for
the word2vec document vectors from B5. In B7 you will keep this exact model and this exact training
loop and just swap the toy matrix for the real word2vec features.

## What you will practice

1. **Layers** - `nn.Linear`, `nn.ReLU`, `nn.Dropout`, and reading their parameter shapes.
2. **Models** - subclassing `nn.Module` (`__init__` + `forward`) and `nn.Sequential`.
3. **Loss and optimizer** - `nn.CrossEntropyLoss` (raw logits, integer labels) and `Adam`.
4. **Batching** - `TensorDataset` and `DataLoader`.
5. **The training loop** - the six lines that train every model in this course.

## Why this matters for the chatbot

The end goal of the course is a fine-tuned transformer in a Gradio chatbot. The classification head
that DistilBERT fine-tunes in C9 is a `nn.Linear` layer trained with `nn.CrossEntropyLoss`: it is
exactly the toy model you build here, and the HuggingFace trainer runs exactly the loop you write
here. Learn it once on a toy problem; reuse it on real features in B7 and on DistilBERT in C9.

Runtime: about 60 to 90 minutes for all exercises.

<!-- Provenance: FROM-OLD Frameworks/PyTorch_Exercises.ipynb cell b168f35b. Change: keep the setup heading; add the B4/B5 carry-forward note and the Colab restart-runtime note the sister notebooks use. -->

## Section 0 - Environment setup

Run the next two cells first. Colab already ships PyTorch, so we only pin `numpy<2` to stay
consistent with the gensim and scipy versions used in B5. Because Colab preinstalls numpy 2.x, you
must restart the runtime after the install cell (Runtime -> Restart runtime), then run the import
cell. We reuse the same `SEED = 42` and `device` idiom you set up in B4, and we bring back the
`TensorDataset` and `DataLoader` imports that B4 deferred to this notebook.

In [ ]:
# Provenance: FROM-OLD Frameworks/PyTorch_Exercises.ipynb cell 4450bef2.
# Change: drop the torch/torchvision install (Colab already has a numpy-2-compatible torch 2.x);
# pin numpy<2; add the restart comment.

# Colab already has torch; we only pin numpy<2 for consistency with B5 (gensim/scipy).
# After this cell runs, restart the runtime (Runtime -> Restart runtime), then run the imports.
!pip install -q "numpy<2"

# Download the small English spaCy pipeline (model weights, separate from pip).
!python -m spacy download en_core_web_sm

# TextBlob/NLTK tokenizer data (punkt_tab) for sentence/word tokenization.
import nltk
nltk.download('punkt_tab')


In [ ]:
# Provenance: FROM-OLD Frameworks/PyTorch_Exercises.ipynb cell a7f4f1ed.
# Change: keep seeds and device detection; drop torch.nn.functional as F and torchvision usage;
# ADD from torch.utils.data import TensorDataset, DataLoader (the import B4 deferred to B6).

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# ---- reproducibility (same SEED as B4/B5) ----
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# ---- device: a torch.device object, same idiom as B4 ----
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version : {torch.__version__}")
print(f"Using device    : {device}")
print("Setup complete. nn.Module, loss, optimizer, and DataLoader were promised in B4 - here they are.")

In [ ]:
# Provenance: FROM-OLD Frameworks/PyTorch_Exercises.ipynb cells 180ca89c + 34c4581a (merged).
# Change: replace the MNIST image preview with a make_classification toy matrix preview; use 100
# features so the shape matches B5's 100-d doc_vector output, making the B7 swap literal.

# Toy dataset: a float feature matrix that mimics B5's word2vec document vectors.
# In B5 each text became a 100-dim vector (doc_vector). Here we fake 100-dim vectors with
# make_classification so you can focus on the model and the training loop. In B7 you delete
# this cell and load the real word2vec X, y instead.
from sklearn.datasets import make_classification

N_FEATURES = 100   # same width as B5's doc_vector output (glove-wiki-gigaword-100)
N_CLASSES  = 2     # binary, like the SST-2 sentiment task in Part C

X_np, y_np = make_classification(
    n_samples=2000,
    n_features=N_FEATURES,
    n_informative=20,
    n_redundant=10,
    n_classes=N_CLASSES,
    random_state=SEED,
)
X_np = X_np.astype(np.float32)   # features must be float32 for nn.Linear
y_np = y_np.astype(np.int64)     # labels must be int64 (long) for CrossEntropyLoss

print(f"X shape : {X_np.shape}   (samples, features) - features stand in for word2vec dims")
print(f"y shape : {y_np.shape}   class labels in {{0, 1}}")
print(f"X dtype : {X_np.dtype} | y dtype : {y_np.dtype}")

# Visualize the first two feature columns, colored by class (just to see structure exists)
plt.figure(figsize=(5, 4))
plt.scatter(X_np[:, 0], X_np[:, 1], c=y_np, cmap='coolwarm', s=8, alpha=0.5)
plt.xlabel('feature 0'); plt.ylabel('feature 1')
plt.title('Toy feature matrix (2 of 100 dims)')
plt.tight_layout(); plt.show()

<!-- Provenance: FROM-OLD Frameworks/PyTorch_Exercises.ipynb cell 64ec4402. Change: keep the Linear/ReLU/Dropout rows; drop the Conv2d row (no images); reframe shapes around a feature vector instead of a 28x28 image. -->

## Section 1 - Creating layers

Every layer in PyTorch lives in `torch.nn`. For a model that classifies feature vectors you need
three building blocks:

| Layer | What it does | Key arguments |
|-------|--------------|---------------|
| `nn.Linear(in, out)` | Fully connected: `y = x @ W.T + b` | `in_features`, `out_features` |
| `nn.ReLU()` | Element-wise `max(0, x)` | none |
| `nn.Dropout(p)` | Randomly zeroes activations during training | `p` = drop probability |

You recognize `nn.Linear` from B4: it is the matrix multiply plus broadcasted bias add you wrote by
hand. Layers are objects: create one, then call it like a function.

```python
layer = nn.Linear(100, 64)   # create: 100 inputs -> 64 outputs
y = layer(x)                 # call: runs the forward pass
```

A layer tracks its own parameters (weights and biases). You never touch those arrays directly; the
optimizer updates them through `optimizer.step()`. `nn.Linear(in, out)` stores `weight` with shape
`(out, in)` and `bias` with shape `(out,)`.

In [ ]:
# Provenance: FROM-OLD Frameworks/PyTorch_Exercises.ipynb cell 0b5c67c0.
# Change: drop the Conv2d block and the fake-image batch; demo Linear, ReLU, Dropout on a feature
# vector of width N_FEATURES.

# Demo: create the three layers and inspect their behavior on a feature vector

# 1. Linear: maps 100 features -> 64 hidden units
fc = nn.Linear(in_features=N_FEATURES, out_features=64)
print("nn.Linear(100, 64)")
print(f"  weight shape : {fc.weight.shape}   # (out_features, in_features) = (64, 100)")
print(f"  bias shape   : {fc.bias.shape}     # (64,)")

# Run one fake feature vector through it (batch of 1)
one_vec = torch.randn(1, N_FEATURES)
print(f"  input  shape : {one_vec.shape}  -> output shape : {fc(one_vec).shape}")

# 2. ReLU: no parameters, just max(0, x)
relu = nn.ReLU()
demo = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])
print(f"\nnn.ReLU() on {demo.tolist()} -> {relu(demo).tolist()}")

# 3. Dropout: zeroes a fraction of activations in train mode, no-op in eval mode
drop = nn.Dropout(p=0.2)
ones = torch.ones(10)
drop.train()
print(f"\nnn.Dropout(0.2) train mode: {drop(ones).tolist()}")
drop.eval()
print(f"nn.Dropout(0.2) eval mode : {drop(ones).tolist()}  (no-op)")

In [ ]:
# Provenance: FROM-OLD Frameworks/PyTorch_Exercises.ipynb cells 0909e3c7 + 3225d9cc (merged).
# Change: retarget tasks to feature-dim layers; drop the Conv2d task; keep the verification block.

# Lab 1: create and configure layers
# Build the three layers a feature-vector classifier needs.

# 1. A fully connected layer that maps the 100 input features to 64 hidden units.
# nn.Linear(in_features, out_features): the weight has shape (out, in), so (64, 100) here.
fc_layer = nn.Linear(N_FEATURES, 64)

# 2. A ReLU activation (no arguments). It is stateless: just max(0, x) applied elementwise.
relu_layer = nn.ReLU()

# 3. A dropout layer that drops 30 percent of activations during training.
# The argument p is the DROP probability, so p=0.3 zeroes 30 percent of activations in train mode.
dropout_layer = nn.Dropout(p=0.3)

# Common mistake: passing the keep probability (0.7) instead of the drop probability (0.3),
# or forgetting that ReLU/Dropout take no in/out feature sizes - only Linear does.

# --- Verification (do not modify below this line) ---
if fc_layer is not None:
    print(f"fc_layer weight shape : {fc_layer.weight.shape}   (expected torch.Size([64, 100]))")
if relu_layer is not None:
    test = torch.tensor([-1.0, 0.0, 1.0])
    print(f"relu_layer(-1,0,1)    : {relu_layer(test).tolist()}   (expected [0.0, 0.0, 1.0])")
if dropout_layer is not None:
    print(f"dropout_layer.p       : {dropout_layer.p}   (expected 0.3)")

<!-- Provenance: FROM-OLD Frameworks/PyTorch_Exercises.ipynb cells d726877f + 344dddd9 + 92c96a2c (merged). Change: merge the Sequential theory and the nn.Module theory; trace shapes on a (batch, features) tensor, not an image; drop the dual-branch motivation in favor of a plain MLP. Class renamed to SentimentMLP (hidden 128, dropout 0.3) so B7 reuses the same class shape. -->

## Section 2 - Building a model with `nn.Module`

A model is a stack of layers wired together. There are two ways to express one.

For a straight chain, `nn.Sequential` is the quickest:

```python
model = nn.Sequential(
    nn.Linear(100, 128),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(128, 2),   # 2 raw logits, one per class
)
```

For anything you want to name, inspect, or extend, subclass `nn.Module`. This is the form the rest
of the course uses, because it is exactly how HuggingFace models are written. We call the class
`SentimentMLP`, because this is the exact class B7 reuses on real word2vec features.

```python
class SentimentMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_classes, dropout=0.3):
        super().__init__()                 # MANDATORY: registers the layers as parameters
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.act = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, n_classes)

    def forward(self, x):                   # define the computation
        h = self.act(self.fc1(x))
        h = self.dropout(h)
        return self.fc2(h)                  # raw logits, NO softmax
```

Two rules that beginners get wrong:

1. Always call `super().__init__()`. Skip it and your layers are not registered, so the optimizer
   never sees them and the model never learns.
2. Return raw logits from `forward`. Do NOT apply softmax. The loss function in Section 3 applies
   log-softmax internally; doing it twice silently breaks training.

Trace the shapes: input `(batch, 100)` -> `fc1` -> `(batch, 128)` -> `fc2` -> `(batch, 2)`.


**Diagram: the SentimentMLP forward pass.** Input features flow through fc1, ReLU, dropout, fc2 to raw logits.

![SentimentMLP forward pass](https://raw.githubusercontent.com/axel-sirota/ml_and_nlp/main/exercises/2-Build-It/diagrams/pytorch-nn-basics/mlp-module-forward.png)

In [ ]:
# Provenance: FROM-OLD Frameworks/PyTorch_Exercises.ipynb cells aacd3462 + bcc94f9d (merged).
# Change: replace the dual-branch model with a single clean SentimentMLP (the class B7 reuses
# same shape as B7: 100 -> 128, dropout 0.3); run a shape check and count parameters; print architecture.

# Demo: define an nn.Module classifier and inspect it

class SentimentMLP(nn.Module):
    """A 2-layer MLP: input_dim -> hidden_dim -> n_classes. Returns raw logits.

    This is the exact class B7 reuses on real word2vec features (same name, same shape:
    Linear -> ReLU -> Dropout -> Linear, hidden_dim=128, dropout=0.3).
    """

    def __init__(self, input_dim, hidden_dim=128, n_classes=2, dropout=0.3):
        super().__init__()                          # registers child layers as parameters
        self.fc1     = nn.Linear(input_dim, hidden_dim)
        self.act     = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2     = nn.Linear(hidden_dim, n_classes)

    def forward(self, x):
        h = self.act(self.fc1(x))                   # (B, input_dim) -> (B, hidden_dim)
        h = self.dropout(h)
        return self.fc2(h)                          # (B, hidden_dim) -> (B, n_classes), raw logits

demo_model = SentimentMLP(input_dim=N_FEATURES, hidden_dim=128, n_classes=N_CLASSES)

# Shape check on a fake batch of 8 feature vectors
probe = torch.randn(8, N_FEATURES)
logits = demo_model(probe)
print(f"Input  : {probe.shape}")
print(f"Output : {logits.shape}   (expected (8, 2) raw logits)")

print("\nArchitecture:")
print(demo_model)

n_params = sum(p.numel() for p in demo_model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {n_params:,}")


In [ ]:
# Provenance: FROM-OLD Frameworks/PyTorch_Exercises.ipynb cells 13a6e0dd + dc1bc2c3 (merged).
# Change: replace the residual-block skeleton with the SentimentMLP skeleton (the class B7 reuses
# same shape as B7: 100 -> 128, dropout 0.3); keep a verification block.

# Lab 2: implement the model class B7 will reuse on real word2vec features.

class SentimentMLP(nn.Module):
    """A 2-layer MLP classifier. forward(x) returns raw logits of shape (B, n_classes)."""

    def __init__(self, input_dim, hidden_dim=128, n_classes=2, dropout=0.3):
        super().__init__()  # keep this line - it registers your layers
        # First fully connected layer: maps input_dim features to hidden_dim units.
        self.fc1     = nn.Linear(input_dim, hidden_dim)
        # A ReLU activation.
        self.act     = nn.ReLU()
        # A dropout layer that drops 30 percent of activations.
        self.dropout = nn.Dropout(dropout)
        # Second fully connected layer: maps hidden_dim units to n_classes outputs.
        self.fc2     = nn.Linear(hidden_dim, n_classes)

    def forward(self, x):
        # Apply fc1, then the activation, then dropout, then fc2.
        # Return raw logits (do not apply softmax).
        h = self.act(self.fc1(x))    # (B, input_dim) -> (B, hidden_dim), then nonlinearity
        h = self.dropout(h)          # randomly zero 30 percent of hidden units (train mode only)
        return self.fc2(h)           # (B, hidden_dim) -> (B, n_classes); RAW logits, no softmax
        # Common mistakes: forgetting super().__init__() (layers never register, model never learns),
        # or applying softmax here (CrossEntropyLoss applies log-softmax itself, so it would be double).

# --- Verification (do not modify) ---
try:
    m = SentimentMLP(input_dim=N_FEATURES, hidden_dim=128, n_classes=N_CLASSES)
    out = m(torch.randn(4, N_FEATURES))
    print(f"Output shape : {out.shape}   (expected torch.Size([4, 2]))")
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"Parameters   : {n:,}   (expected {N_FEATURES*128 + 128 + 128*N_CLASSES + N_CLASSES:,})")
    print("SentimentMLP looks good - this is the exact class you reuse in B7.")
except Exception as e:
    print(f"Not ready yet: {e}")


<!-- Provenance: FROM-NEW. Change: G1 - collapse the Lab 2 safety-net answer into a reveal so it does not show the SentimentMLP body before the student tries. The next code cell is a silent runtime guard only. -->

The next code cell is a silent guard: if your Lab 2 `SentimentMLP` is missing or not working, it
quietly installs a working fallback so the rest of the notebook runs. It does not print the answer.
If you are stuck on Lab 2 and want to see one correct `SentimentMLP`, reveal it below.

<details>
<summary>Stuck? Reveal the safety-net</summary>

```python
class SentimentMLP(nn.Module):
    """SentimentMLP: input_dim -> 128 -> n_classes, dropout 0.3, raw logits."""

    def __init__(self, input_dim, hidden_dim=128, n_classes=2, dropout=0.3):
        super().__init__()
        self.fc1     = nn.Linear(input_dim, hidden_dim)
        self.act     = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2     = nn.Linear(hidden_dim, n_classes)

    def forward(self, x):
        h = self.act(self.fc1(x))
        h = self.dropout(h)
        return self.fc2(h)
```
</details>

In [ ]:
# Safety-net (silent guard): make sure a usable SentimentMLP exists before the back half runs.
# This only fires if your Lab 2 class is missing or broken; it does not print the solution. If you
# want to see a correct SentimentMLP, use the "Reveal the safety-net" toggle in the cell above.

def _sentimentmlp_is_usable():
    try:
        _probe = SentimentMLP(input_dim=N_FEATURES, hidden_dim=128, n_classes=N_CLASSES)
        _out = _probe(torch.randn(2, N_FEATURES))
        return tuple(_out.shape) == (2, N_CLASSES)
    except Exception:
        return False

if not _sentimentmlp_is_usable():
    print("SentimentMLP missing or not working yet - installing a fallback so later cells run.")

    class SentimentMLP(nn.Module):  # silent fallback; see the reveal above for the explained version
        def __init__(self, input_dim, hidden_dim=128, n_classes=2, dropout=0.3):
            super().__init__()
            self.fc1     = nn.Linear(input_dim, hidden_dim)
            self.act     = nn.ReLU()
            self.dropout = nn.Dropout(dropout)
            self.fc2     = nn.Linear(hidden_dim, n_classes)

        def forward(self, x):
            h = self.act(self.fc1(x))
            h = self.dropout(h)
            return self.fc2(h)

    print("Fallback SentimentMLP installed.")
else:
    print("SentimentMLP is defined and working - using your Lab 2 class.")


<!-- Provenance: FROM-OLD Frameworks/PyTorch_Exercises.ipynb cells a7115e5d + 3f1f07e8 (merged). Change: drop the custom-loss material; keep only loss choice (CrossEntropyLoss), optimizer choice (Adam/AdamW), and merge in the six-line-loop framing. -->

## Section 3 - Loss, optimizer, and batching

Three pieces turn a model into a trained model: a loss, an optimizer, and a loop.

**Loss.** For classification, use `nn.CrossEntropyLoss`. Two things to memorize, because both are
common silent bugs:

- It takes RAW LOGITS, not softmax outputs. It applies log-softmax internally.
- Its targets are INTEGER CLASS INDICES (`torch.long`), shape `(N,)`, not one-hot vectors. Pass
  floats or one-hot and you get a dtype error or wrong gradients.

**Optimizer.** `torch.optim.Adam(model.parameters(), lr=1e-3)` is the default for most tasks.
`AdamW` adds proper weight decay and is the default for transformers (you will meet it again in C9).

**Batching.** Wrap your tensors in a `TensorDataset`, then a `DataLoader` shuffles and batches them:

```python
ds = TensorDataset(X_tensor, y_tensor)
loader = DataLoader(ds, batch_size=64, shuffle=True)
```

**The loop.** Keras gives you `model.fit`. PyTorch makes you write the loop, which is the same six
lines every time:

```python
for xb, yb in loader:
    optimizer.zero_grad()        # 1. clear stale gradients
    logits = model(xb)           # 2. forward pass
    loss = loss_fn(logits, yb)   # 3. compute loss
    loss.backward()              # 4. backpropagate
    optimizer.step()             # 5. update weights
# (6. the for-loop itself is the iteration over batches)
```

The number one beginner bug is forgetting `optimizer.zero_grad()`. PyTorch accumulates gradients by
default, so skipping it sums this batch's gradient onto every previous batch's, and the loss slowly
explodes. Make `zero_grad` the first line of the loop, always.

**Diagram: the batching pipeline.** TensorDataset pairs features with labels; DataLoader shuffles and serves batches to the loop.

![Batching pipeline](https://raw.githubusercontent.com/axel-sirota/ml_and_nlp/main/exercises/2-Build-It/diagrams/pytorch-nn-basics/data-pipeline.png)

In [ ]:
# Provenance: FROM-OLD Frameworks/PyTorch_Exercises.ipynb cell ce46960e.
# Change: drop the sine dataset and custom-MSE comparison; demo CrossEntropyLoss on raw logits with
# integer targets, and show one optimizer step shrinking the loss. Convert toy numpy arrays to
# tensors here (float32 features, long labels) so the dtype rules are visible.

# Demo: loss + optimizer, one step at a time

# Convert the toy numpy arrays to tensors with the REQUIRED dtypes
X_tensor = torch.tensor(X_np, dtype=torch.float32)   # features: float32
y_tensor = torch.tensor(y_np, dtype=torch.long)      # labels: long (int64) class indices
print(f"X_tensor dtype : {X_tensor.dtype} | y_tensor dtype : {y_tensor.dtype}")

model     = SentimentMLP(input_dim=N_FEATURES, hidden_dim=128, n_classes=N_CLASSES).to(device)
loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# One forward pass on a small batch
xb = X_tensor[:128].to(device)
yb = y_tensor[:128].to(device)

logits = model(xb)
print(f"logits shape   : {logits.shape}   (batch, n_classes) - RAW, no softmax")
loss_before = loss_fn(logits, yb)
print(f"loss before    : {loss_before.item():.4f}")

# One optimization step
optimizer.zero_grad()
loss_before.backward()
optimizer.step()

loss_after = loss_fn(model(xb), yb)
print(f"loss after 1 step: {loss_after.item():.4f}   (should be lower)")

In [ ]:
# Provenance: FROM-NEW.
# Change: new lab for the toy arc. Student builds the TensorDataset / DataLoader and creates the
# loss and optimizer for their SentimentMLP.

# Lab 3: prepare the data pipeline and the training ingredients.

BATCH_SIZE = 64
LR = 1e-3

# 1. Wrap X_tensor and y_tensor in a dataset, then build a loader that
#    shuffles and serves batches of BATCH_SIZE.
# TensorDataset pairs each feature row with its label; DataLoader handles shuffling and batching.
train_ds     = TensorDataset(X_tensor, y_tensor)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

# 2. Instantiate your SentimentMLP (100 -> 128 -> 2) and move it to device.
# .to(device) moves the parameters to GPU if one is available (same idiom you will use in C9).
model = SentimentMLP(input_dim=N_FEATURES, hidden_dim=128, n_classes=N_CLASSES).to(device)

# 3. The loss function for classification (raw logits + integer labels).
loss_fn = nn.CrossEntropyLoss()

# 4. An Adam optimizer over the model's parameters at learning rate LR.
# Always pass model.parameters() so the optimizer knows which tensors to update.
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# Common mistake: forgetting shuffle=True (the model sees the same batch order every epoch),
# or building the optimizer BEFORE the model exists so it has no parameters to update.

# --- Verification (do not modify) ---
if train_loader is not None:
    xb, yb = next(iter(train_loader))
    print(f"batch X shape : {xb.shape}   (expected (64, 100))")
    print(f"batch y shape : {yb.shape}   (expected (64,))")
    print(f"batch y dtype : {yb.dtype}   (expected torch.int64)")
if model is not None and loss_fn is not None and optimizer is not None:
    sample = loss_fn(model(xb.to(device)), yb.to(device))
    print(f"sample loss   : {sample.item():.4f}   (a single float, ready to train)")

<!-- Provenance: FROM-NEW. Change: G1 - collapse the Lab 3 safety-net answer (the four training ingredients) into a reveal so it does not show before the student tries. The next code cell is a silent runtime guard only. -->

The next code cell is a silent guard: if any of `train_loader`, `model`, `loss_fn`, or `optimizer`
is still missing, it quietly fills it with a working default so Lab 4 runs. It does not print the
answer. If you are stuck on Lab 3 and want to see one correct setup, reveal it below.

<details>
<summary>Stuck? Reveal the safety-net</summary>

```python
train_loader = DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=64, shuffle=True)
model        = SentimentMLP(input_dim=N_FEATURES, hidden_dim=128, n_classes=N_CLASSES).to(device)
loss_fn      = nn.CrossEntropyLoss()
optimizer    = torch.optim.Adam(model.parameters(), lr=1e-3)
```
</details>

In [ ]:
# Safety-net (silent guard): make sure the four training ingredients exist before Lab 4 runs.
# Each block only fires if its name is still missing; it does not print the solution. If you want to
# see one correct Lab 3 setup, use the "Reveal the safety-net" toggle in the cell above.

if 'train_loader' not in globals() or train_loader is None:
    train_loader = DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=64, shuffle=True)

if 'model' not in globals() or model is None:
    model = SentimentMLP(input_dim=N_FEATURES, hidden_dim=128, n_classes=N_CLASSES).to(device)

if 'loss_fn' not in globals() or loss_fn is None:
    loss_fn = nn.CrossEntropyLoss()

if 'optimizer' not in globals() or optimizer is None:
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("Training ingredients ready: train_loader, model, loss_fn, optimizer.")


<!-- Provenance: FROM-NEW. Change: new short theory cell isolating the full epoch loop and the eval pattern (model.eval(), torch.no_grad(), accuracy idiom), so the lab that follows is pure recall. -->

## Section 4 - Train for several epochs and evaluate

One pass over all the batches is one epoch. You repeat for several epochs and watch the loss fall.
After training, you measure accuracy in evaluation mode:

- `model.train()` before the training batches: turns dropout ON.
- `model.eval()` before evaluating: turns dropout OFF so predictions are deterministic.
- Wrap evaluation in `with torch.no_grad():` so PyTorch does not build the autograd graph. This is
  faster and uses less memory.

Accuracy for a classification model is the fraction of correct predictions. Take the argmax over the
class dimension and compare to the labels:

```python
preds = logits.argmax(dim=1)
acc = (preds == y).float().mean().item()
```

That is the whole evaluation. In B7 you run this exact code on real word2vec features and compare
the accuracy to the LogisticRegression baseline from B5.

**Diagram: the six-line training loop.** zero_grad, forward, loss, backward, step, repeated over every batch.

![Six-line training loop](https://raw.githubusercontent.com/axel-sirota/ml_and_nlp/main/exercises/2-Build-It/diagrams/pytorch-nn-basics/training-loop.png)

In [ ]:
# Provenance: FROM-OLD Frameworks/PyTorch_Exercises.ipynb cell e191b15f.
# Change: drop the sine regression; train the classifier on the toy DataLoader; track and plot the
# loss curve; print accuracy each epoch using the argmax idiom.

# Demo: train the classifier end to end on the toy data

EPOCHS = 15

demo_model = SentimentMLP(input_dim=N_FEATURES, hidden_dim=128, n_classes=N_CLASSES).to(device)
demo_loss_fn   = nn.CrossEntropyLoss()
demo_optimizer = torch.optim.Adam(demo_model.parameters(), lr=1e-3)

demo_ds     = TensorDataset(X_tensor, y_tensor)
demo_loader = DataLoader(demo_ds, batch_size=64, shuffle=True)

losses = []
for epoch in range(EPOCHS):
    demo_model.train()                      # dropout ON
    epoch_loss = 0.0
    for xb, yb in demo_loader:
        xb, yb = xb.to(device), yb.to(device)
        demo_optimizer.zero_grad()          # 1. clear grads
        logits = demo_model(xb)             # 2. forward
        loss = demo_loss_fn(logits, yb)     # 3. loss
        loss.backward()                     # 4. backward
        demo_optimizer.step()               # 5. update
        epoch_loss += loss.item()
    avg = epoch_loss / len(demo_loader)
    losses.append(avg)

    # evaluation
    demo_model.eval()                       # dropout OFF
    with torch.no_grad():
        all_logits = demo_model(X_tensor.to(device))
        preds = all_logits.argmax(dim=1).cpu()
        acc = (preds == y_tensor).float().mean().item()
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d}/{EPOCHS} - loss: {avg:.4f} | train acc: {acc:.4f}")

plt.figure(figsize=(6, 3))
plt.plot(range(1, EPOCHS+1), losses, marker='o')
plt.xlabel('epoch'); plt.ylabel('avg loss'); plt.title('Training loss')
plt.tight_layout(); plt.show()

In [ ]:
# Provenance: FROM-OLD Frameworks/PyTorch_Exercises.ipynb cells e99ac07b + 13bd65df (merged).
# Change: retarget from MNIST to the toy classifier; the student fills the six loop lines and the
# eval forward/argmax; verification asserts accuracy crosses a threshold.

# Lab 4: write the training loop for YOUR model from Lab 3.
# Reuse train_loader, model, loss_fn, optimizer from Lab 3.

LAB_EPOCHS = 15

for epoch in range(LAB_EPOCHS):
    model.train()
    epoch_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        # Step 1: clear the gradients left over from the previous batch.
        # PyTorch accumulates gradients by default, so this MUST be the first line of the loop.
        optimizer.zero_grad()

        # Step 2: run the forward pass to get logits.
        logits = model(xb)

        # Step 3: compute the loss between logits and labels.
        loss = loss_fn(logits, yb)

        # Step 4: backpropagate.
        loss.backward()

        # Step 5: take one optimizer step.
        optimizer.step()

        # Silent rescue: if the five steps above are still empty, run one real step here so the
        # notebook keeps training instead of crashing on the line below. Fires only while your
        # loop is unfinished; once you fill in the five steps, this becomes a no-op.
        if loss is None:
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()

        epoch_loss += loss.item()

    # --- evaluation (provided) ---
    model.eval()
    with torch.no_grad():
        # Forward pass over the full dataset, then take the predicted class per row.
        eval_logits = model(X_tensor.to(device))     # forward on all rows (no dropout in eval mode)
        preds = eval_logits.argmax(dim=1)            # argmax over the class dimension -> class index
        # Silent rescue: degrade gracefully if the two lines above are still empty.
        if eval_logits is None:
            eval_logits = model(X_tensor.to(device))
        if preds is None:
            preds = eval_logits.argmax(dim=1)
        acc = (preds.cpu() == y_tensor).float().mean().item()

    avg = epoch_loss / len(train_loader)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d}/{LAB_EPOCHS} - loss: {avg:.4f} | train acc: {acc:.4f}")

# Common mistake: dropping optimizer.zero_grad() (gradients pile up and the loss explodes),
# or calling argmax over dim 0 (across the batch) instead of dim 1 (across the classes).

# --- Verification (guarded: warns instead of halting so the notebook keeps running) ---
if acc is not None:
    print(f"\nFinal training accuracy: {acc:.4f}   (expected above 0.85)")
    if acc > 0.85:
        print("Training loop works. This is the loop that fine-tunes DistilBERT in C9.")
    else:
        print("WARNING: train accuracy is below 0.85. Check your loop wiring (zero_grad first,")
        print("         forward, loss, backward, step) - but the notebook will keep running.")

<!-- Provenance: FROM-NEW. Change: G1 - Lab 4 safety-net reveal. The training loop above already self-rescues silently if you leave the five steps empty; this collapsed block shows one correct loop only if you choose to reveal it. -->

The Lab 4 cell above already self-rescues: if you leave the five steps empty it runs one real step
per batch silently so the loop still trains and the notebook does not crash. If you are stuck and
want to see one correct version of the six-line loop and the eval block, reveal it below.

<details>
<summary>Stuck? Reveal the safety-net</summary>

```python
for epoch in range(LAB_EPOCHS):
    model.train()
    epoch_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()          # 1. clear stale gradients
        logits = model(xb)             # 2. forward pass
        loss = loss_fn(logits, yb)     # 3. compute loss
        loss.backward()                # 4. backpropagate
        optimizer.step()               # 5. update weights
        epoch_loss += loss.item()

    model.eval()
    with torch.no_grad():
        eval_logits = model(X_tensor.to(device))
        preds = eval_logits.argmax(dim=1)
        acc = (preds.cpu() == y_tensor).float().mean().item()
```
</details>

In [ ]:
# Provenance: FROM-OLD Frameworks/PyTorch_Exercises.ipynb cells 137d60f0 + b3859256 (merged).
# Change: simplify the transfer end-to-end lab into a clean end-to-end demo function train_model
# that wraps the whole pattern, so B7 can call it on real features. Print final accuracy.

# The whole pattern as one reusable function. B7 calls this on real word2vec features.

def train_model(X, y, input_dim, hidden_dim=128, n_classes=2, dropout=0.3,
                epochs=15, lr=1e-3, batch_size=64):
    """Train a SentimentMLP on (X, y) tensors. Returns the trained model and final accuracy.

    Same SentimentMLP and same six-line loop B7 reuses on the word2vec features.
    """
    ds = TensorDataset(X, y)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True)
    net = SentimentMLP(input_dim, hidden_dim, n_classes, dropout).to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)

    for epoch in range(epochs):
        net.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = loss_fn(net(xb), yb)
            loss.backward()
            optimizer.step()

    net.eval()
    with torch.no_grad():
        preds = net(X.to(device)).argmax(dim=1).cpu()
        acc = (preds == y).float().mean().item()
    return net, acc

trained_net, final_acc = train_model(X_tensor, y_tensor, input_dim=N_FEATURES)
print(f"End-to-end toy accuracy: {final_acc:.4f}")
print("In B7: call train_model on the word2vec X, y and beat the B5 LogisticRegression baseline.")


<!-- Provenance: FROM-NEW. Change: new instructions cell for the end-to-end lab plus the labelled stretch (validation split), keeping the three-tier lab structure. -->

### Lab 5 - Run it end to end, then make it honest (stretch)

**Core task.** You already have `train_model`. Call it on the toy tensors and confirm you get a
trained model and an accuracy above 0.85. Then pass a larger `hidden_dim` (say 256) and re-run: note
whether more capacity helps on this toy data.

**Stretch (for fast finishers).** Training accuracy alone is misleading: a model can memorize the
training set. Split the data into a train part and a held-out validation part, train only on the
train part, and report accuracy on BOTH each epoch. Track the best validation accuracy you see.
This is the single most important habit before C9, where you fine-tune on a train split and report
on a validation split. The starter is in the next cell.

In [ ]:
# Provenance: FROM-NEW.
# Change: new lab. Core line calls train_model. Stretch builds a train/val split and tracks best val
# accuracy.

# Lab 5 core: train end to end on the toy tensors.
# train_model wraps the whole pattern and returns (trained_model, accuracy).
core_net, core_acc = train_model(X_tensor, y_tensor, input_dim=N_FEATURES)
# Silent rescue: run the core training if the line above is still empty, so the stretch below
# always has a trained reference and does not silently no-op. A no-op once you fill it in.
if core_acc is None:
    core_net, core_acc = train_model(X_tensor, y_tensor, input_dim=N_FEATURES)
if core_acc is not None:
    print(f"Core accuracy: {core_acc:.4f}   (expected above 0.85)")

# --- Stretch: train/validation split with best-val tracking ---
# 1. Choose a split point that holds out the last 20 percent of the rows for validation.
n_total = X_tensor.shape[0]
n_val   = int(0.2 * n_total)          # 20 percent of the rows, as an integer count

# 2. Slice tensors into train and validation parts (first part trains, last n_val rows validate).
# The first (n_total - n_val) rows train; the final n_val rows are held out for validation.
X_train = X_tensor[:n_total - n_val]
y_train = y_tensor[:n_total - n_val]
X_val   = X_tensor[n_total - n_val:]
y_val   = y_tensor[n_total - n_val:]

# Silent rescue: if the split above is still empty, build a default last-20-percent split so the
# stretch loop still runs end to end instead of being skipped. A no-op once you fill it in.
if n_val is None:
    n_val = int(0.2 * n_total)
if X_train is None or y_train is None or X_val is None or y_val is None:
    X_train = X_tensor[:n_total - n_val]
    y_train = y_tensor[:n_total - n_val]
    X_val   = X_tensor[n_total - n_val:]
    y_val   = y_tensor[n_total - n_val:]

# Common mistake: validating on rows the model also trained on (data leakage), which makes the
# validation accuracy look better than the model really is.

if n_val is not None and X_val is not None:
    train_loader_s = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
    net_s = SentimentMLP(N_FEATURES, 128, N_CLASSES).to(device)
    loss_fn_s = nn.CrossEntropyLoss()
    opt_s = torch.optim.Adam(net_s.parameters(), lr=1e-3)

    best_val = 0.0
    for epoch in range(15):
        net_s.train()
        for xb, yb in train_loader_s:
            xb, yb = xb.to(device), yb.to(device)
            opt_s.zero_grad()
            loss_fn_s(net_s(xb), yb).backward()
            opt_s.step()
        net_s.eval()
        with torch.no_grad():
            tr_acc  = (net_s(X_train.to(device)).argmax(1).cpu() == y_train).float().mean().item()
            val_acc = (net_s(X_val.to(device)).argmax(1).cpu()   == y_val).float().mean().item()
        best_val = max(best_val, val_acc)
    print(f"train acc: {tr_acc:.4f} | val acc: {val_acc:.4f} | best val: {best_val:.4f}")
    print("If train acc is much higher than val acc, the model is overfitting - see the homework.")

<!-- Provenance: FROM-OLD Frameworks/PyTorch_Exercises.ipynb cells 4c1e03a8 + d87a3d58 + 8e1adeac + 7be26845 (merged). Change: fold the entire freezing section (theory, demo, lab) into one async homework cell, retheme to the chatbot transfer-learning story, and add a second bonus on overfitting with dropout + weight decay. -->

## Homework extension (async, deeper)

These are async, production-oriented, and point straight at C9. Do them in this notebook after class.

**Homework 1 - Freeze a backbone, train a fresh head, then fine-tune (the C9 pattern).**

Real transfer learning loads a pretrained model, freezes it, snaps a fresh head on top, and trains
only the head. Then it optionally unfreezes and fine-tunes at a low learning rate. You will simulate
this with two MLPs.

1. Build a model with a `backbone` (`nn.Sequential` of a couple of `Linear` + `ReLU` layers) and a
   separate `head` (`nn.Linear`) as two attributes in one `nn.Module`.
2. Freeze the backbone: loop over `model.backbone.parameters()` and set `requires_grad = False`.
3. Build the optimizer over trainable params only:
   `torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)`.
4. Count trainable vs frozen params with `sum(p.numel() for p in model.parameters() if p.requires_grad)`
   and confirm only the head is trainable.
5. Train a few epochs, record accuracy. Then unfreeze everything (`requires_grad = True` for all
   params), rebuild the optimizer at `lr=1e-4`, fine-tune one more epoch, and compare.

This is exactly what C9 does: DistilBERT is the frozen-then-fine-tuned backbone, and a `nn.Linear`
classifier is the fresh head trained with `nn.CrossEntropyLoss`.

**Homework 2 - See overfitting, then fix it.**

1. Take a tiny slice (say 200 rows) of the toy data and a train/val split.
2. Train a wide MLP for many epochs. Watch train accuracy approach 1.0 while validation accuracy
   stalls or drops: that gap is overfitting.
3. Now regularize: raise `nn.Dropout` p, and swap `Adam` for `torch.optim.AdamW(..., weight_decay=0.01)`.
   AdamW applies decoupled weight decay and is the default optimizer for transformers in C9.
4. Re-run and confirm the train/val gap shrinks.

Write two or three sentences on what changed and why this matters before fine-tuning a large model.

**Diagram: freeze then fine-tune (the C9 pattern).** Freeze the backbone, train a fresh head, then unfreeze and fine-tune at a low learning rate.

![Freeze then fine-tune](https://raw.githubusercontent.com/axel-sirota/ml_and_nlp/main/exercises/2-Build-It/diagrams/pytorch-nn-basics/transfer-learning-freeze.png)

<!-- Provenance: FROM-OLD Frameworks/PyTorch_Exercises.ipynb cell 10a2e758. Change: retarget the recap table to the four kept sections; drop the Conv2d/dual-branch/custom-loss rows; add the explicit bridge to B7 and to C9. Class named SentimentMLP to match B7 verbatim. -->

## Recap and what is next

Here is a summary of what you learned: you built and trained a neural network end to end. The pieces:

| Section | PyTorch concept |
|---------|-----------------|
| 1 | Layers: `nn.Linear`, `nn.ReLU`, `nn.Dropout` and their parameter shapes |
| 2 | Models: `nn.Module` subclassing (`__init__` + `forward`), `nn.Sequential` |
| 3 | Loss + optimizer + batching: `nn.CrossEntropyLoss`, `Adam`, `TensorDataset`, `DataLoader` |
| 4 | The six-line training loop: `zero_grad -> forward -> loss -> backward -> step`, plus eval |

Three rules to keep:

1. `optimizer.zero_grad()` is the first line of the loop. Forgetting it is the most common bug.
2. `nn.CrossEntropyLoss` takes raw logits and integer (long) labels. Never apply softmax first.
3. `model.train()` for training, `model.eval()` plus `torch.no_grad()` for evaluation.

### Bridge to B7 (the stopper)

You trained on a toy `make_classification` matrix. In B7 you delete that matrix and load the real
word2vec document vectors from B5 (the saved `X`, `y`). You keep this exact `SentimentMLP` and this
exact loop, and your job is to beat the LogisticRegression baseline you measured in B5. That is the
"embeddings as features" pattern: words become vectors (B5), vectors feed an MLP (B6), the MLP
classifies (B7).

### Bridge to C9 (the chatbot)

The classifier `DistilBertForSequenceClassification` fine-tunes in C9 is an encoder plus a
`nn.Linear` head trained with `nn.CrossEntropyLoss` - the same head you built here. The HuggingFace
trainer runs the same `zero_grad -> forward -> loss -> backward -> step` loop you just wrote. You now
know the engine that trains the chatbot.